# EDA — Disputas de transacciones (cargos no reconocidos)
### Factored AI & Data Hackathon 2026 — Equipo DataMastersGT

## Qué datos se usaron y por qué

El dataset completo del reto (`factored-datathon-2026-s3-157725502942-us-east-2-an`) tiene ~19M filas en 13 tablas,
con particionado Hive por fecha (`year=/month=/day=`). Para este análisis exploratorio **no se bajó el dataset completo**,
se construyó una muestra dirigida al caso de uso elegido (disputas por cargo no reconocido), priorizando cobertura
temporal donde más importa:

| Tabla | Cobertura descargada | Justificación |
|---|---|---|
| `customers`, `products`, `branches`, `service_agents` | 100% (tablas dimensión, son chicas) | Se necesitan completas para joins |
| `complaints` | Año completo 2026 (diario) | Es la tabla objetivo del proyecto — necesitamos todo el detalle |
| `call_center_interactions` | Año completo 2026 (diario) | Tabla clave del flujo de disputas (origen de los casos) |
| `call_transcripts` | Mayo–junio 2026 completo (diario) | Para extraer intents/idioma reales; no se necesita todo el año para eso |
| `satisfaction_surveys` | Enero–mediados feb diario + muestra semanal resto del año | Tabla de apoyo (contexto de CSAT), no es el foco |
| `transactions` | Muestra semanal abril–junio 2026 | Tabla de apoyo/contexto (volumen muy alto, no es el foco del flujo de disputas) |

Esta estrategia de muestreo (denso en las tablas núcleo del caso de uso, disperso en las de contexto) se adoptó
para mantener el análisis manejable sin perder las dos tablas que de verdad sostienen el proyecto: `complaints` y
`call_center_interactions`.

**Fuente:** bucket S3 oficial del hackathon, acceso de solo lectura provisto en el Data Dictionary del reto.
**Fecha de extracción:** 30 de septiembre de 2026.


## Setup

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

con = duckdb.connect()
base = "../hackathon-data"

con.execute(f"CREATE VIEW customers AS SELECT * FROM read_csv_auto('{base}/customers.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW products AS SELECT * FROM read_csv_auto('{base}/products.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW branches AS SELECT * FROM read_csv_auto('{base}/branches.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW agents AS SELECT * FROM read_csv_auto('{base}/service_agents.csv', ignore_errors=true)")
con.execute(f"CREATE VIEW complaints AS SELECT * FROM read_csv_auto('{base}/complaints/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW cci AS SELECT * FROM read_csv_auto('{base}/call_center_interactions/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW surveys AS SELECT * FROM read_csv_auto('{base}/satisfaction_surveys/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW txns AS SELECT * FROM read_csv_auto('{base}/transactions/**/*.csv', ignore_errors=true, union_by_name=true)")
con.execute(f"CREATE VIEW transcripts AS SELECT * FROM read_csv_auto('{base}/call_transcripts/**/*.csv', ignore_errors=true, union_by_name=true)")

print("Vistas creadas OK")

## 1. Tamaño de la muestra por tabla

In [ ]:
con.execute('''
SELECT 'customers' tabla, count(*) n FROM customers
UNION ALL SELECT 'products', count(*) FROM products
UNION ALL SELECT 'complaints', count(*) FROM complaints
UNION ALL SELECT 'call_center_interactions', count(*) FROM cci
UNION ALL SELECT 'satisfaction_surveys', count(*) FROM surveys
UNION ALL SELECT 'transactions (muestra)', count(*) FROM txns
UNION ALL SELECT 'call_transcripts', count(*) FROM transcripts
''').df()

## 2. Hallazgo clave: las disputas de "Transactions" son casi 100% cargos no reconocidos

Esto valida directamente el caso de uso elegido: el dataset ya viene modelado casi exclusivamente
para el escenario de **cargo no reconocido**, no es una categoría genérica.

In [ ]:
con.execute('''
SELECT subcategory, count(*) n, round(avg(claimed_amount),0) avg_amt,
       round(100.0*sum(case when sla_breached then 1 else 0 end)/count(*),1) pct_breach,
       round(avg(resolution_days),1) avg_days,
       round(avg(resolution_satisfaction),2) avg_sat
FROM complaints WHERE category='Transactions' GROUP BY subcategory ORDER BY n DESC
''').df()

## 3. Complaints — panorama general por categoría

In [ ]:
con.execute('''
SELECT category, count(*) n, round(avg(claimed_amount),1) avg_amt,
       round(100.0*sum(case when sla_breached then 1 else 0 end)/count(*),1) pct_sla_breach,
       round(avg(resolution_days),1) avg_days
FROM complaints GROUP BY category ORDER BY n DESC
''').df()

## 4. Hallazgo clave: el triage actual no protege nada

El % de incumplimiento de SLA es prácticamente el mismo sin importar la prioridad asignada al caso
(Critical, High, Medium, Low). Esto es la evidencia central que justifica un agente de re-priorización
basado en datos en vez de una asignación manual/fija.

In [ ]:
con.execute('''
SELECT priority, count(*) n,
       round(100.0*sum(case when sla_breached then 1 else 0 end)/count(*),1) pct_breach,
       round(avg(resolution_days),1) avg_days
FROM complaints GROUP BY priority ORDER BY priority
''').df()

## 5. Estado y canal de recepción de las disputas

In [ ]:
con.execute('''
SELECT status, count(*) n FROM complaints GROUP BY status ORDER BY n DESC
''').df()

In [ ]:
con.execute('''
SELECT reception_channel, count(*) n FROM complaints GROUP BY reception_channel ORDER BY n DESC
''').df()

## 6. Trazabilidad rota: 0% de las disputas de Transactions están ligadas a la llamada que las originó

In [ ]:
con.execute('''
SELECT origin_interaction_id IS NOT NULL as tiene_interaccion_origen, count(*) n
FROM complaints WHERE category='Transactions' GROUP BY 1
''').df()

## 7. Call Center Interactions — tasa de resolución y escalación por tipo de motivo

Las interacciones de tipo "Queja" resuelven en primer contacto a menos de la mitad de tasa que el resto
(43.3% vs. 90%+), coherente con que son las que más terminan escalando a `complaints`.

In [ ]:
con.execute('''
SELECT reason_category, count(*) n,
       round(100.0*sum(case when was_resolved then 1 else 0 end)/count(*),1) pct_resuelto_1er_contacto,
       round(100.0*sum(case when was_escalated then 1 else 0 end)/count(*),1) pct_escalado
FROM cci GROUP BY reason_category ORDER BY n DESC
''').df()

## 8. Transcripts — confirma la limitación de idioma del dataset

El 100% de las transcripciones detectadas en la muestra están en español. No hay datos en portugués
en el dataset — se documenta como limitación conocida del *dato*, no del sistema: los casos de prueba
en portugués para el agente se generan manualmente por el equipo.

In [ ]:
con.execute('''
SELECT detected_language, count(*) n FROM transcripts GROUP BY detected_language ORDER BY n DESC
''').df()

### Intención detectada en los transcripts

El campo `detected_intents` que ya trae el dataset es casi todo `"consulta_general"` — demasiado genérico
para enrutar un caso. Esto justifica construir un clasificador de intención propio (vs. ese dato crudo).

In [ ]:
con.execute('''
SELECT detected_intents, count(*) n FROM transcripts GROUP BY detected_intents ORDER BY n DESC LIMIT 10
''').df()

## 9. Satisfacción del cliente (surveys)

In [ ]:
con.execute('''
SELECT survey_type, round(avg(main_score),2) avg_score, min(main_score) min_score, max(main_score) max_score, count(*) n
FROM surveys GROUP BY survey_type ORDER BY n DESC
''').df()

## 10. Calidad de datos — limitaciones a tener en cuenta en el pipeline

No se ocultan, se documentan: el reto premia reconocerlas y manejarlas explícitamente.

In [ ]:
con.execute('''
SELECT
  count(*) total_complaints,
  sum(case when claimed_amount is null then 1 else 0 end) nulls_claimed_amount,
  round(100.0*sum(case when claimed_amount is null then 1 else 0 end)/count(*),1) pct_nulls_amount,
  sum(case when resolution_date is null then 1 else 0 end) nulls_resolution_date,
  round(100.0*sum(case when resolution_date is null then 1 else 0 end)/count(*),1) pct_nulls_resdate
FROM complaints
''').df()

Los nulos en `claimed_amount` y `resolution_date` son consistentes con que ~68% de los casos
siguen `In Process`/`Open` (no cerrados todavía) — es realista para un banco en operación, pero el
pipeline del agente debe manejar esos nulos sin asumir que siempre hay monto o fecha de cierre.

## Conclusiones — qué justifican estos hallazgos para el proyecto

1. **Problema real y medible**: 2,093 disputas/año de cargo no reconocido, $2,458 promedio reclamado, 22% incumple SLA, 15 días promedio de resolución.
2. **El triage manual no funciona** (SLA breach ~20-22% parejo en todas las prioridades) → justifica un modelo de re-priorización basado en riesgo, comparado contra ese baseline de reglas fijas.
3. **El campo de intención del dataset es demasiado genérico** (95% "consulta_general") → justifica un clasificador de intención propio.
4. **La trazabilidad interacción→disputa está rota** (0% ligadas) → el sistema la resuelve por diseño.
5. **El dataset es 100% español** → los casos de portugués para el agente se generan manualmente y se documentan como limitación del dato, no del sistema.
